# 🚀 Training with Unsloth
### D4BL Tutorial 3 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/03_training_with_unsloth.ipynb)

**What you'll learn:** How to fine-tune Qwen2.5-3B with LoRA adapters using D4BL's actual training configuration. You'll run real training on Colab's free T4 GPU.

**Time:** ~10 min (quick mode) / ~25 min (full training) | **Prerequisites:** Google account with Colab GPU access | **Dependencies:** unsloth, transformers, trl

⚠️ **Important:** This notebook requires a GPU runtime. Go to Runtime → Change runtime type → T4 GPU.

In [ ]:
# === CONFIGURATION ===
# Set to False for full 7-epoch training run (~25 min on T4)
QUICK_MODE = True  # True = 10 steps only (~2 min), good for learning

# Install Unsloth (optimized for Colab T4)
!pip install -q unsloth transformers datasets trl

## What is LoRA?

LoRA (Low-Rank Adaptation) lets you fine-tune a large model by training only a small set of adapter weights — typically less than 1% of the total parameters. The original model stays frozen.

**Why this matters for equity work:** LoRA makes fine-tuning possible on free hardware. You don't need a $10,000 GPU or a cloud computing budget. A community organization can train a model on Colab's free T4 GPU in under 30 minutes.

D4BL uses rank 16, which means each adapter is about 98K parameters — 0.003% of the 3B base model.

In [ ]:
import torch
from unsloth import FastLanguageModel

# Load Qwen2.5-3B — D4BL's base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    dtype=None,  # auto-detect
    load_in_4bit=True,  # saves VRAM on free Colab
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuring LoRA

These are D4BL's actual LoRA settings from Sprint 2.5. Each parameter matters:

| Parameter | Value | Why |
|-----------|-------|-----|
| `r` (rank) | 16 | Sweet spot: enough capacity for structured JSON output without overfitting |
| `lora_alpha` | 32 | 2x rank — standard scaling that keeps learning stable |
| `lora_dropout` | 0.05 | Light regularization to prevent memorizing training examples |
| `target_modules` | q_proj, k_proj, v_proj, o_proj, etc. | All attention + MLP layers — the model needs to learn new output patterns |

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({trainable/total:.4%} of total)")

## Preparing the Dataset

This is where Sprint 2.5's critical fix lives. The original Sprint 2 training manually concatenated ChatML strings — and the model failed to produce structured JSON. The fix: use `tokenizer.apply_chat_template()`, which produces the exact token boundaries the model expects.

**Wrong way (Sprint 2):**
```python
text = "<|im_start|>system\n" + system + "<|im_end|>\n..."  # Manual string concat — BROKEN
```

**Right way (Sprint 2.5):**
```python
text = tokenizer.apply_chat_template(messages, tokenize=False)  # Native template — WORKS
```

This single change took the integration test pass rate from 3/11 to 11/11.

In [ ]:
import json

from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an AI assistant trained to support data justice and racial equity research "
    "following the Data for Black Lives (D4BL) methodology.\n\n"
    "Core principles:\n"
    "1. Center communities most impacted by structural racism.\n"
    "2. Name structural causes of racial disparities.\n"
    "3. Connect findings to policy levers.\n"
    "4. Acknowledge data limitations.\n"
    "5. Make analysis accessible to communities.\n\n"
    "Respond with ONLY valid JSON."
)

def make_pair(question, response_dict):
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": json.dumps(response_dict)},
    ]}

SAMPLE_TRAINING_DATA = [
    make_pair("What's the homeownership gap between Black and white families in Georgia?", {
        "entities": ["Georgia"], "search_queries": ["homeownership rate racial disparity Georgia", "Black white homeownership gap Georgia"],
        "data_sources": ["census_acs"], "community_framing": {"detected": False, "issue_domain": "housing", "structural_frame": None}
    }),
    make_pair("Why are so many people in my neighborhood getting asthma?", {
        "entities": ["neighborhood"], "search_queries": ["asthma prevalence environmental racism", "air quality racial disparity", "industrial pollution proximity minority communities"],
        "data_sources": ["cdc_places", "epa_ejscreen"], "community_framing": {"detected": True, "issue_domain": "health", "structural_frame": "environmental_racism"}
    }),
    make_pair("Compare poverty rates by race in Alabama", {
        "entities": ["Alabama"], "search_queries": ["poverty rate by race Alabama", "Black white poverty gap Alabama"],
        "data_sources": ["census_acs"], "community_framing": {"detected": False, "issue_domain": "economic", "structural_frame": None}
    }),
    make_pair("Why do they keep closing hospitals in our part of town?", {
        "entities": ["neighborhood", "hospitals"], "search_queries": ["hospital closures Black communities", "healthcare access racial disparity", "medical desert rural urban"],
        "data_sources": ["cdc_places"], "community_framing": {"detected": True, "issue_domain": "health", "structural_frame": "disinvestment"}
    }),
    make_pair("Show me education funding data for Mississippi", {
        "entities": ["Mississippi"], "search_queries": ["education funding per pupil Mississippi", "school funding racial disparity Mississippi"],
        "data_sources": ["doe", "census_acs"], "community_framing": {"detected": False, "issue_domain": "education", "structural_frame": None}
    }),
    make_pair("How come our water never gets fixed but they built a new park across town?", {
        "entities": ["water infrastructure", "parks"], "search_queries": ["water infrastructure racial disparity", "municipal investment racial bias", "environmental justice water quality"],
        "data_sources": ["epa_ejscreen", "census_acs"], "community_framing": {"detected": True, "issue_domain": "environment", "structural_frame": "municipal_disinvestment"}
    }),
    make_pair("What is the unemployment rate for Black workers in Georgia?", {
        "entities": ["Georgia"], "search_queries": ["unemployment rate Black workers Georgia", "racial employment gap Georgia"],
        "data_sources": ["bls", "census_acs"], "community_framing": {"detected": False, "issue_domain": "employment", "structural_frame": None}
    }),
    make_pair("Why are Black men being arrested at higher rates in this county?", {
        "entities": ["county"], "search_queries": ["arrest rate racial disparity", "Black male incarceration rates", "over-policing Black communities"],
        "data_sources": ["fbi_ucr", "police_violence"], "community_framing": {"detected": True, "issue_domain": "criminal_justice", "structural_frame": "over_policing"}
    }),
    make_pair("Median household income trends in Alabama by race", {
        "entities": ["Alabama"], "search_queries": ["median household income trend Alabama race", "income inequality Alabama historical"],
        "data_sources": ["census_acs"], "community_framing": {"detected": False, "issue_domain": "economic", "structural_frame": None}
    }),
    make_pair("Nobody in this neighborhood can afford to buy a house anymore", {
        "entities": ["neighborhood", "housing"], "search_queries": ["homeownership barriers Black families", "housing affordability racial disparity", "discriminatory lending practices"],
        "data_sources": ["census_acs", "hud"], "community_framing": {"detected": True, "issue_domain": "housing", "structural_frame": "discriminatory_lending"}
    }),
    make_pair("What does the lead paint data show for minority communities?", {
        "entities": ["minority communities"], "search_queries": ["lead paint exposure racial disparity", "lead poisoning children minority neighborhoods"],
        "data_sources": ["epa_ejscreen", "cdc_places"], "community_framing": {"detected": False, "issue_domain": "environment", "structural_frame": None}
    }),
    make_pair("Our kids can't even play outside because of the air quality", {
        "entities": ["children", "air quality"], "search_queries": ["air quality index minority neighborhoods", "PM2.5 exposure children racial disparity", "outdoor recreation access environmental justice"],
        "data_sources": ["epa_ejscreen", "cdc_places"], "community_framing": {"detected": True, "issue_domain": "health", "structural_frame": "environmental_racism"}
    }),
    make_pair("Show diabetes prevalence data by race for Southern states", {
        "entities": ["Southern states"], "search_queries": ["diabetes prevalence race Southern states", "chronic disease racial disparity South"],
        "data_sources": ["cdc_places"], "community_framing": {"detected": False, "issue_domain": "health", "structural_frame": None}
    }),
    make_pair("Why do Black women keep dying having babies in this state?", {
        "entities": ["state"], "search_queries": ["maternal mortality Black women", "pregnancy related deaths racial disparity", "obstetric racism"],
        "data_sources": ["cdc_places"], "community_framing": {"detected": True, "issue_domain": "health", "structural_frame": "medical_racism"}
    }),
    make_pair("Compare eviction rates by race in Atlanta metro area", {
        "entities": ["Atlanta"], "search_queries": ["eviction rate racial disparity Atlanta", "housing instability Black renters Atlanta"],
        "data_sources": ["hud", "census_acs"], "community_framing": {"detected": False, "issue_domain": "housing", "structural_frame": None}
    }),
    make_pair("Every time they build something new it's never for us", {
        "entities": ["development", "community"], "search_queries": ["gentrification displacement Black communities", "urban development racial equity", "community benefit agreements"],
        "data_sources": ["census_acs", "hud"], "community_framing": {"detected": True, "issue_domain": "housing", "structural_frame": "gentrification"}
    }),
    make_pair("What are the high school graduation rates by race in Mississippi?", {
        "entities": ["Mississippi"], "search_queries": ["high school graduation rate race Mississippi", "education attainment racial gap Mississippi"],
        "data_sources": ["doe", "census_acs"], "community_framing": {"detected": False, "issue_domain": "education", "structural_frame": None}
    }),
    make_pair("They put all the waste sites near where we live", {
        "entities": ["waste sites", "residential areas"], "search_queries": ["hazardous waste sites racial demographics", "environmental racism toxic facilities", "TSDF proximity minority communities"],
        "data_sources": ["epa_ejscreen"], "community_framing": {"detected": True, "issue_domain": "environment", "structural_frame": "environmental_racism"}
    }),
    make_pair("Police use of force statistics by race nationally", {
        "entities": ["national"], "search_queries": ["police use of force racial disparity national", "excessive force statistics by race"],
        "data_sources": ["police_violence", "fbi_ucr"], "community_framing": {"detected": False, "issue_domain": "criminal_justice", "structural_frame": None}
    }),
    make_pair("How come there aren't any good jobs around here?", {
        "entities": ["local area", "employment"], "search_queries": ["job access racial disparity", "employment desert minority neighborhoods", "occupational segregation"],
        "data_sources": ["bls", "census_acs"], "community_framing": {"detected": True, "issue_domain": "employment", "structural_frame": "occupational_segregation"}
    }),
]

# Format using the tokenizer's native chat template (Sprint 2.5 fix)
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(SAMPLE_TRAINING_DATA)
dataset = dataset.map(format_example)

print(f"Training examples: {len(dataset)}")
print("\nFirst example (formatted):")
print(dataset[0]["text"][:300] + "...")

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    num_train_epochs=1 if QUICK_MODE else 7,
    max_steps=10 if QUICK_MODE else -1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    output_dir="outputs",
    optim="adamw_8bit",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

print(f"Training mode: {'QUICK (10 steps)' if QUICK_MODE else 'FULL (7 epochs)'}")
print("Starting training...")
stats = trainer.train()
print(f"\nDone! Final loss: {stats.training_loss:.4f}")

In [ ]:
import os

model.save_pretrained("d4bl-query-parser-adapter")
tokenizer.save_pretrained("d4bl-query-parser-adapter")

adapter_size = sum(
    os.path.getsize(os.path.join("d4bl-query-parser-adapter", f))
    for f in os.listdir("d4bl-query-parser-adapter")
)
print("Adapter saved to: d4bl-query-parser-adapter/")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print("Base model size: ~6.2 GB (FP16)")
print(f"Adapter is {adapter_size / 6.2e9:.4%} of the base model")

## ✏️ Exercise

1. Change `QUICK_MODE = False` and run the full training. Watch the loss curve — it should decrease steadily.
2. Try changing the LoRA rank: set `r=8` or `r=32` in the LoRA config cell. How does it affect adapter size and training speed?
3. Check the training loss at different steps. If it plateaus early, the model may need more data or epochs.

## Summary

You just fine-tuned a 3B parameter language model on free hardware. The key ingredients: LoRA for parameter efficiency, `apply_chat_template()` for correct formatting, and D4BL's methodology embedded in every training example.

**Next:** [Notebook 4 — Testing Your Model](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/04_testing_your_model.ipynb) → Load your trained model and see how it compares to the base.